In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Встановлення рівня оптимізації транспілятора

<details>
<summary><b>Версії пакетів</b></summary>

Код на цій сторінці розроблено з використанням наведених нижче залежностей.
Рекомендуємо використовувати ці або новіші версії.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Реальні квантові пристрої схильні до шумів і помилок вентилів, тому оптимізація схем для зменшення їхньої глибини та кількості вентилів може суттєво покращити результати виконання цих схем.
Функція [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) має один обов'язковий позиційний аргумент — `optimization_level`, який визначає, скільки зусиль транспілятор витрачає на оптимізацію схем. Цей аргумент може бути цілим числом з множини: 0, 1, 2 або 3.
Вищі рівні оптимізації генерують більш оптимізовані схеми ціною збільшення часу компіляції.
У наведеній нижче таблиці пояснюються оптимізації, що виконуються при кожному налаштуванні.

<table>
  <thead>
    <tr>
      <th>Рівень оптимізації</th>
      <th>Опис</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>0</td>
      <td>
        Без оптимізації: зазвичай використовується для характеризації апаратного забезпечення
        - Базовий переклад
        - Розміщення/Маршрутизація: `TrivialLayout`, де вибираються ті самі номери фізичних кубітів, що й віртуальних, та вставляються SWAP-вентилі для забезпечення роботи (з використанням `SabreSwap`)
      </td>
    </tr>
    <tr>
      <td>1</td>
      <td>
        Легка оптимізація:
        -   Розміщення/Маршрутизація: спочатку виконується спроба розміщення з `TrivialLayout`. Якщо потрібні додаткові SWAP, знаходиться розміщення з мінімальною кількістю SWAP за допомогою `SabreSwap`, а потім `VF2LayoutPostLayout` намагається вибрати найкращі кубіти у графі.
        -   `InverseCancellation`
        -   Оптимізація однокубітних вентилів
      </td>
    </tr>
    <tr>
      <td>2</td>
      <td>
        Середня оптимізація:
          - Розміщення/Маршрутизація: рівень оптимізації 1 (без trivial) + евристична оптимізація з більшою глибиною пошуку та кількістю спроб функції оптимізації. Оскільки `TrivialLayout` не використовується, немає спроби використати однакові номери фізичних і віртуальних кубітів.
        -   `CommutativeCancellation`
      </td>
    </tr>
    <tr>
      <td>3</td>
      <td>
        Висока оптимізація:
        - Рівень оптимізації 2 + подальша евристична оптимізація розміщення/маршрутизації з більшими зусиллями/кількістю спроб
        - Ресинтез двокубітних блоків за допомогою [KAK-розкладання Картана](https://arxiv.org/abs/quant-ph/0507171).
        - Проходи, що порушують унітарність:
          * `OptimizeSwapBeforeMeasure`: переміщує вимірювання для уникнення SWAP
          * `RemoveDiagonalGatesBeforeMeasure`: видаляє вентилі перед вимірюваннями, що не впливають на результати вимірювань
      </td>
    </tr>
  </tbody>
</table>

## Рівень оптимізації в дії
Оскільки двокубітні вентилі зазвичай є найзначнішим джерелом помилок, ми можемо приблизно оцінити «апаратну ефективність» транспіляції, підрахувавши кількість двокубітних вентилів у результуючій схемі.
Тут ми спробуємо різні рівні оптимізації на вхідній схемі, що складається з випадкового унітарного оператора, за яким слідує SWAP-вентиль.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator, random_unitary

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qc = QuantumCircuit(2)
qc.append(rand_U, range(2))
qc.swap(0, 1)
qc.draw("mpl", style="iqp")

<Image src="../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg" alt="Output of the previous code cell" />

![Вивід попередньої комірки коду](../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg)

Ми будемо використовувати симульований бекенд `FakeSherbrooke` в наших прикладах. Спочатку виконаємо транспіляцію з рівнем оптимізації 0.

In [2]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg" alt="Output of the previous code cell" />

![Вивід попередньої комірки коду](../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg)

Транспільована схема містить шість двокубітних вентилів ECR.

Повторимо для рівня оптимізації 1:

In [3]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg" alt="Output of the previous code cell" />

![Вивід попередньої комірки коду](../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg)

Транспільована схема все ще містить шість вентилів ECR, але кількість однокубітних вентилів зменшилася.

Повторимо для рівня оптимізації 2:

In [4]:
pass_manager = generate_preset_pass_manager(
    optimization_level=2, backend=backend, seed_transpiler=12345
)
qc_t2_exact = pass_manager.run(qc)
qc_t2_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg" alt="Output of the previous code cell" />

![Вивід попередньої комірки коду](../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg)

Це дає ті самі результати, що й рівень оптимізації 1. Зверни увагу, що підвищення рівня оптимізації не завжди має ефект.

Повторимо ще раз, з рівнем оптимізації 3:

In [5]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=12345
)
qc_t3_exact = pass_manager.run(qc)
qc_t3_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg" alt="Output of the previous code cell" />

![Вивід попередньої комірки коду](../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg)

Тепер є лише три вентилі ECR. Ми отримуємо цей результат тому, що на рівні оптимізації 3 Qiskit намагається повторно синтезувати двокубітні блоки вентилів, і будь-який двокубітний вентиль можна реалізувати не більш ніж трьома вентилями ECR. Ми можемо отримати ще менше вентилів ECR, якщо встановимо `approximation_degree` менше 1, дозволяючи транспілятору робити наближення, що можуть вносити певну похибку в розкладання вентилів (дивись [Поширено використовувані параметри транспіляції](common-parameters#approximation-degree)):

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    approximation_degree=0.99,
    backend=backend,
    seed_transpiler=12345,
)
qc_t3_approx = pass_manager.run(qc)
qc_t3_approx.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg" alt="Output of the previous code cell" />

![Вивід попередньої комірки коду](../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg)

Ця схема містить лише два вентилі ECR, але є наближеною. Щоб зрозуміти, наскільки її дія відрізняється від точної схеми, ми можемо обчислити точність між унітарним оператором, що реалізує ця схема, та точним унітарним. Перед обчисленням ми спочатку зводимо транспільовану схему, яка містить 127 кубітів, до схеми лише з активними кубітами — їх двоє.

In [7]:
import numpy as np


def trace_to_fidelity_2q(trace: float) -> float:
    return (4.0 + trace * trace.conjugate()) / 20.0


# Reduce circuits down to 2 qubits so they are easy to simulate
qc_t3_exact_small = QuantumCircuit.from_instructions(qc_t3_exact)
qc_t3_approx_small = QuantumCircuit.from_instructions(qc_t3_approx)

# Compute the fidelity
exact_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_exact_small).adjoint().data, UU))
)
approx_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_approx_small).adjoint().data, UU))
)
print(
    f"Synthesis fidelity\nExact: {exact_fid:.3f}\nApproximate: {approx_fid:.3f}"
)

Synthesis fidelity
Exact: 1.000+0.000j
Approximate: 0.992+0.000j


Adjusting the optimization level can change other aspects of the circuit too, not just the number of ECR gates. For examples of how setting optimization level changes the layout, see [Representing quantum computers](./represent-quantum-computers).

## Next steps

<Admonition type="tip" title="Recommendations">
    - To learn more about the `generate_preset_passmanager` function, start with the [Transpilation default settings and configuration options](defaults-and-configuration-options) topic.
    - Continue learning about transpilation with the [Transpiler stages](transpiler-stages) topic.
    - Try the [Compare transpiler settings](/docs/guides/circuit-transpilation-settings) guide.
    - Try the [Build repetition codes](/docs/tutorials/repetition-codes) tutorial.
    - See the [Transpile API documentation.](/docs/api/qiskit/transpiler)
</Admonition>